# IMPORT LIBRARY

In [ ]:
import datetime
import mysql.connector
from mysql.connector import Error
from mysql.connector import errorcode
import time

from pytz import timezone

import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date

import importlib, importlib.util

import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine, text
import psycopg2

import requests
import json

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

import pymysql
pd.set_option('display.max_columns', None)


# CONFIG DB

In [ ]:
# Import Config Data
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dl_dwh    = module_from_file("config_dw",       "../db_config/config_db_datawarehouse.py")
config_ksj_mbase = module_from_file("config_mbase",    "../db_config/config_db_ksj_link_mbase.py")
config_jiwaplus  = module_from_file("config_jiwaplus", "../db_config/config_db_jiwaplus.py")


In [ ]:
# ==========================================================================================
# NAMA DATABASE
# ------------------------------------------------------------------------------------------
# CATATAN: config_db_ksj_link_mbase.py menulis d_notifications = 'd_notifications' (JAMAK),
# padahal nama DB aslinya 'd_notification' (TUNGGAL). Notebook ini tidak memakainya,
# tapi dicatat di sini supaya konsisten dengan etl_ksj_on_demand_rider_availability.ipynb.
#
# Ketiga database di bawah ini ada di HOST yang sama (db-ksj.jiwa.prod) tapi merupakan
# DATABASE TERPISAH -> tidak bisa di-JOIN lewat SQL, penggabungannya dilakukan di pandas.
# ==========================================================================================
DB_NOTIFICATION = 'd_notification'
DB_DISPATCHER   = config_ksj_mbase.d_jiwa_ondemand    # 'jiwa-order-dispatcher'
DB_TRANSACTION  = config_ksj_mbase.d_transaction      # 'd_transaction'

print('DB_DISPATCHER  :', DB_DISPATCHER)
print('DB_TRANSACTION :', DB_TRANSACTION)


In [ ]:
def dl_dwh_db_conn(db_name):
    return pymysql.connect(
        host=config_dl_dwh.hostname,
        db=db_name,
        user=config_dl_dwh.username,
        password=config_dl_dwh.password,
        port=3306
    )


In [ ]:
def ksj_mbase_db_conn(db_name):
    """Koneksi ke host db-ksj.jiwa.prod (read-only user bi_cursor_jiwa)."""
    return psycopg2.connect(
        host=config_ksj_mbase.hostname,
        database=db_name,
        user=config_ksj_mbase.username,
        password=config_ksj_mbase.password,
        port='5432'
    )


In [ ]:
def jiwaplus_db_conn():
    """Koneksi ke host db.jiwa.prod, database jiwaplus.

    Host & user BEDA dengan ksj_mbase_db_conn -- ini database checkout JiwaPlus,
    dipakai untuk data yang tidak ada di database KSJ (nama/telepon customer,
    status transaksi versi JiwaPlus, dan data voucher membership).
    """
    return psycopg2.connect(
        host=config_jiwaplus.hostname,
        database=config_jiwaplus.database,
        user=config_jiwaplus.username,
        password=config_jiwaplus.password,
        port='5432'
    )


In [ ]:
def run_query(db_name, sql, params=None):
    """Helper: eksekusi query ke host ksj lalu balikin DataFrame."""
    connection = ksj_mbase_db_conn(db_name)
    cursor = connection.cursor()

    cursor.execute(sql, params)

    record = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()

    return df


In [ ]:
def run_query_jiwaplus(sql, params=None):
    """Helper: eksekusi query ke database jiwaplus lalu balikin DataFrame."""
    connection = jiwaplus_db_conn()
    cursor = connection.cursor()

    cursor.execute(sql, params)

    record = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()

    return df


# PARAMETER

In [ ]:
# ==========================================================================================
# ROUTING API
# ------------------------------------------------------------------------------------------
# Tidak ada satupun kolom jarak rute yang tersimpan di database KSJ manapun (sudah dicek
# seluruh kolom bernama %distance%). Yang tersedia hanya dispatch_blast_riders.distance_in_meter,
# dan itu jarak GARIS LURUS (haversine) rider -> titik delivery, bukan jarak rute jalan.
# Jadi actual route distance memang harus dihitung sendiri lewat routing API.
#
# ROUTE_PROVIDER:
#   'OSRM'   -> router.project-osrm.org, gratis, tanpa API key. Default.
#               Catatan: demo server OSRM hanya menyediakan profil 'driving'
#               (profil cycling/foot mengembalikan angka identik alias alias saja).
#               Untuk motor, angkanya cenderung over-estimate -> konservatif.
#   'GOOGLE' -> butuh GOOGLE_MAPS_API_KEY diisi. Lebih akurat & stabil, tapi berbayar.
#   'NONE'   -> matikan routing, actual_route_distance_m diisi NULL.
#
# Kalau API gagal/timeout, kolomnya diisi NULL dan route_source='FAILED'
# -- ETL TIDAK BOLEH mati gara-gara API pihak ketiga.
# ==========================================================================================
ROUTE_PROVIDER      = 'OSRM'
OSRM_BASE           = 'https://router.project-osrm.org/route/v1/driving'
GOOGLE_MAPS_API_KEY = None
ROUTE_TIMEOUT       = 15      # detik per request
ROUTE_MAX_RETRY     = 2       # percobaan ulang kalau gagal
ROUTE_SLEEP_SECONDS = 0.3     # jeda antar request, biar sopan ke server publik

# Semua kolom timestamp di server masih UTC -> perlu +7 jam supaya jadi WIB.
# PENGECUALIAN: d_transaction.transactions.transaction_date SUDAH WIB.

# Nama tabel target di DWH
TABLE_ORDER_SUMMARY = 'ksj_on_demand_order_summary'

# Tanggal paling awal data on-demand tersedia
DATA_START_DATE = '2026-08-03'

# Voucher "Markas Sejuta Jiwa" -- dipakai untuk flag have_markassajiwa.
# Dicek terhadap CURRENT_DATE (hari ETL ini dijalankan) -- artinya flag ini menunjukkan
# status keanggotaan customer SAAT INI, bukan status di tanggal order historis dibuat.
MARKASSAJIWA_VOUCHER_ID = 96965


# FUNCTION

## Extract

In [ ]:
def get_pending_orders(start_date_str, end_date_str):
    """Satu baris per order on-demand, dari jiwa-order-dispatcher.pending_orders.

    Ini tabel utama (grain tabel summary kita). Alurnya: pembayaran customer sukses
    -> baris masuk ke pending_orders -> sistem blast ke rider.

    Catatan kolom:
      rider_lat / rider_long  = posisi rider yang ACCEPT order (bukan posisi outlet).
                                Sudah diverifikasi identik dengan responded_rider_lat/long
                                milik rider yang accept.
      status                  = ACCEPTED (ada rider terima) / EXPIRED (tidak ada yang terima
                                dalam 6 menit, order auto reject-refund) / CANCELLED.
    """
    query = """
        SELECT
            id                                                          AS pending_order_id,
            jiwaplus_ref_id,
            transaction_id,
            hub_code,
            customer_id,
            status                                                      AS dispatcher_status,
            total_item_qty,
            total_price,
            total_price_after_discount,
            payment_type,
            delivery_lat,
            delivery_lng,
            rider_lat                                                   AS accepted_rider_lat,
            rider_long                                                  AS accepted_rider_lng,
            blast_count,
            accepted_rider_code,
            (created_at   AT TIME ZONE 'Asia/Jakarta')                  AS created_at_wib,
            (accepted_at  AT TIME ZONE 'Asia/Jakarta')                  AS accepted_at_wib,
            (expired_at   AT TIME ZONE 'Asia/Jakarta')                  AS expired_at_wib,
            (cancelled_at AT TIME ZONE 'Asia/Jakarta')                  AS cancelled_at_wib,
            ROUND(EXTRACT(EPOCH FROM (expires_at - created_at)) / 60, 2) AS order_ttl_minutes,
            ROUND(EXTRACT(EPOCH FROM (accepted_at - created_at)), 1)     AS seconds_to_accept
        FROM pending_orders
        WHERE (created_at AT TIME ZONE 'Asia/Jakarta')::date BETWEEN '{0}' AND '{1}'
        ORDER BY id;
    """.format(start_date_str, end_date_str)

    return run_query(DB_DISPATCHER, query)


In [ ]:
def get_blast_rider_metrics(start_date_str, end_date_str):
    """Agregat per order dari dispatch_blast_riders -- berapa rider yang di-ping & responnya apa.

    PENTING: jangan JOIN dispatch_blasts dan dispatch_blast_riders sekaligus di satu query
    lalu di-COUNT, karena hasilnya cartesian product (order dengan 2 blast round akan
    terbaca 2x lipat jumlah ping-nya). Makanya agregasi keduanya dipisah lalu di-merge.

    CATATAN: nearest/avg/median/max_blasted_rider_distance_m TIDAK dihitung di sini lagi.
    Sebelumnya pakai MIN/AVG/MAX(distance_in_meter) -- itu jarak GARIS LURUS yang dihitung
    sistem dispatcher (blast position -> delivery). Sekarang diganti jarak RUTE JALAN (OSRM),
    dihitung per rider di section TRANSFORM (lihat get_blast_rider_positions() di bawah).
    """
    query = """
        SELECT
            r.pending_order_id,
            COUNT(*)                                                AS total_riders_blasted,
            COUNT(*) FILTER (WHERE r.response = 'ACCEPTED')          AS riders_accepted,
            COUNT(*) FILTER (WHERE r.response = 'REJECTED')          AS riders_rejected,
            COUNT(*) FILTER (WHERE r.response = 'TIMEOUT')           AS riders_timeout,
            COUNT(*) FILTER (WHERE r.response IS NULL)               AS riders_no_response,
            COUNT(*) FILTER (WHERE r.wa_status   = 'SENT')           AS riders_wa_sent,
            COUNT(*) FILTER (WHERE r.push_status = 'SENT')           AS riders_push_sent
        FROM dispatch_blast_riders r
        JOIN pending_orders p ON p.id = r.pending_order_id
        WHERE (p.created_at AT TIME ZONE 'Asia/Jakarta')::date BETWEEN '{0}' AND '{1}'
        GROUP BY 1;
    """.format(start_date_str, end_date_str)

    return run_query(DB_DISPATCHER, query)


In [ ]:
def get_blast_round_metrics(start_date_str, end_date_str):
    """Agregat per order dari dispatch_blasts -- berapa ronde blast & radiusnya.

    Sistem melakukan blast tiap 2 menit, maksimal 3 ronde. Radius melebar tiap ronde
    (3000 m -> 7500 m). Kalau sampai ronde terakhir tidak ada yang accept, order EXPIRED.
    """
    query = """
        SELECT
            b.pending_order_id,
            COUNT(*)                    AS total_blast_rounds,
            MAX(b.blast_round)          AS last_blast_round,
            MIN(b.radius_m)             AS first_blast_radius_m,
            MAX(b.radius_m)             AS max_blast_radius_m,
            SUM(b.riders_notified)      AS total_riders_notified,
            (MIN(b.sent_at) AT TIME ZONE 'Asia/Jakarta')  AS first_blast_at_wib,
            (MAX(b.sent_at) AT TIME ZONE 'Asia/Jakarta')  AS last_blast_at_wib
        FROM dispatch_blasts b
        JOIN pending_orders p ON p.id = b.pending_order_id
        WHERE (p.created_at AT TIME ZONE 'Asia/Jakarta')::date BETWEEN '{0}' AND '{1}'
        GROUP BY 1;
    """.format(start_date_str, end_date_str)

    return run_query(DB_DISPATCHER, query)


In [ ]:
def get_accept_context(start_date_str, end_date_str):
    """Untuk order yang diterima: di ronde blast ke berapa rider akhirnya accept,
    dan sampai saat itu sudah di-dispatch ke berapa rider.

    Menjawab pertanyaan: "setiap order yang diterima, rider itu menerimanya setelah
    order di-dispatch ke berapa banyak rider?"
    """
    query = """
        WITH acc AS (
            SELECT r.pending_order_id, MIN(b.blast_round) AS accepted_blast_round
            FROM dispatch_blast_riders r
            JOIN dispatch_blasts b ON b.id = r.blast_id
            WHERE r.response = 'ACCEPTED'
            GROUP BY 1
        )
        SELECT
            r.pending_order_id,
            MIN(a.accepted_blast_round)                                              AS accepted_blast_round,
            COUNT(*) FILTER (WHERE b.blast_round <= a.accepted_blast_round)          AS riders_blasted_until_accept
        FROM dispatch_blast_riders r
        JOIN dispatch_blasts  b ON b.id = r.blast_id
        JOIN acc              a ON a.pending_order_id = r.pending_order_id
        JOIN pending_orders   p ON p.id = r.pending_order_id
        WHERE (p.created_at AT TIME ZONE 'Asia/Jakarta')::date BETWEEN '{0}' AND '{1}'
        GROUP BY 1;
    """.format(start_date_str, end_date_str)

    return run_query(DB_DISPATCHER, query)


In [ ]:
def get_nearest_blasted_rider(start_date_str, end_date_str):
    """Koordinat rider TERDEKAT yang di-blast untuk tiap order.

    Dipakai sebagai titik asal untuk order yang GAGAL (EXPIRED / CANCELLED), karena
    order itu tidak punya rider yang accept sehingga tidak ada titik asal yang pasti.
    Interpretasinya: "rider terdekat yang tersedia saat itu jaraknya berapa".
    """
    query = """
        SELECT DISTINCT ON (r.pending_order_id)
            r.pending_order_id,
            r.rider_code             AS nearest_blasted_rider_code,
            r.blast_rider_lat        AS nearest_blasted_rider_lat,
            r.blast_rider_long       AS nearest_blasted_rider_lng
        FROM dispatch_blast_riders r
        JOIN pending_orders p ON p.id = r.pending_order_id
        WHERE r.distance_in_meter IS NOT NULL
          AND (p.created_at AT TIME ZONE 'Asia/Jakarta')::date BETWEEN '{0}' AND '{1}'
        ORDER BY r.pending_order_id, r.distance_in_meter ASC;
    """.format(start_date_str, end_date_str)

    return run_query(DB_DISPATCHER, query)


In [ ]:
def get_blast_rider_positions(pending_order_ids):
    """Posisi SETIAP rider yang di-blast (bukan agregat 1 baris/order seperti fungsi di atas).

    Dipakai untuk hitung jarak RUTE JALAN (OSRM) per rider individual, lalu di-agregasi
    (min/mean/median/max) di TRANSFORM. Query ini TIDAK memfilter status order -- baik
    order yang ACCEPTED maupun EXPIRED/CANCELLED (rider tidak accept) sama-sama ikut,
    karena jarak semua rider yang di-blast tetap relevan dihitung terlepas hasil ordernya.
    """
    query = """
        SELECT
            pending_order_id,
            rider_code,
            blast_rider_lat,
            blast_rider_long
        FROM dispatch_blast_riders
        WHERE pending_order_id = ANY(%s)
          AND blast_rider_lat  IS NOT NULL
          AND blast_rider_long IS NOT NULL;
    """

    return run_query(DB_DISPATCHER, query, params=(list(pending_order_ids),))


In [ ]:
def get_first_dispatch_reason(pending_order_ids):
    """Alasan dispatch PALING AWAL per order, dari dispatch_records.

    dispatch_records TIDAK punya FK resmi ke pending_orders -- join manual pakai
    order_id (teks, mis. 'SJ-xxxx') = pending_orders.jiwaplus_ref_id.

    Kenapa PALING AWAL (bukan sembarang baris): order yang gagal SELALU dapat baris
    kedua generik "expired - no rider accepted" belakangan -- baris pertama itu yang
    mengandung alasan asli (mis. "no rider with sufficient stock", "no nearby rider").
    Dipakai apa adanya sebagai order_status_detail, tanpa hardcode/rename lagi.
    """
    query = """
        SELECT DISTINCT ON (po.id)
            po.id AS pending_order_id,
            dr.reason AS order_status_detail
        FROM pending_orders po
        JOIN dispatch_records dr ON dr.order_id = po.jiwaplus_ref_id
        WHERE po.id = ANY(%s)
        ORDER BY po.id, dr.created_at ASC;
    """

    return run_query(DB_DISPATCHER, query, params=(list(pending_order_ids),))


In [ ]:
def get_delivery_detail(start_date_str, end_date_str):
    """Detail transaksi & pengantaran dari d_transaction (transactions x on_demand_order_deliveries).

    Kedua tabel ini ada di database yang sama jadi boleh di-JOIN langsung di SQL.
    transaction_type = 'ON_DEMAND' membedakannya dari transaksi KSJ_LINK biasa.

    paid_at      = waktu customer benar-benar bayar (sudah diverifikasi identik ke milidetik
                   dengan pending_orders.order_snapshot->>'paid_at').
    completed_at = waktu transaksi ditutup sistem. Sudah diverifikasi selisihnya dengan
                   on_demand_order_deliveries.delivered_at cuma beberapa milidetik (23 order
                   DELIVERED dicek, semuanya < 10 ms) -- praktis event yang sama.
    rider_completion_lat/lng = posisi rider saat menekan tombol selesai + upload bukti antar.
    """
    query = """
        SELECT
            t.pending_order_id,
            t.id                                            AS trx_id,
            t.transaction_no,
            t.rider_code                                    AS delivery_rider_code,
            t.transaction_status,
            t.transaction_date,
            t.total_item_qty                                AS trx_total_item_qty,
            (t.paid_at      AT TIME ZONE 'Asia/Jakarta')     AS paid_at_wib,
            (t.completed_at AT TIME ZONE 'Asia/Jakarta')     AS trx_completed_at_wib,
            d.delivery_status,
            d.failed_reason,
            d.rider_completion_lat,
            d.rider_completion_lng,
            (d.out_for_delivery_at + INTERVAL '7 hours')     AS out_for_delivery_at_wib,
            (d.delivered_at        + INTERVAL '7 hours')     AS delivered_at_wib
        FROM transactions t
        LEFT JOIN on_demand_order_deliveries d ON d.transaction_id = t.id
        WHERE t.transaction_type = 'ON_DEMAND'
          AND t.transaction_date BETWEEN '{0}'::date - INTERVAL '1 day' AND '{1}'::date + INTERVAL '1 day'
          AND t.pending_order_id IS NOT NULL;
    """.format(start_date_str, end_date_str)

    return run_query(DB_TRANSACTION, query)


In [ ]:
def get_jiwaplus_transaction_detail(transaction_ids):
    """Detail transaksi versi JiwaPlus (customer_name, customer_phone, status checkout).

    JOIN key: jiwaplus.transactions.id_transaction == pending_orders.transaction_id.
    HATI-HATI: ini BUKAN d_transaction.transactions.id (nomor itu beda numbering-nya --
    pending_orders.transaction_id ada di kisaran jutaan yg sama dgn id_transaction jiwaplus,
    sedangkan d_transaction.transactions.id adalah id internal terpisah, sudah dipakai
    sebagai kolom trx_id di get_delivery_detail()).

    transaction_datetime_gmt: TERLEPAS dari namanya, kolom ini isinya WIB, bukan GMT/UTC --
    sudah diverifikasi konsisten +7 jam dari pending_orders.created_at (UTC asli) di 4 sample.
    Jangan digeser +7 jam lagi.
    """
    query = """
        SELECT
            id_transaction,
            transaction_datetime_gmt,
            latest_transaction_status,
            customer_name,
            customer_phone
        FROM transactions
        WHERE id_transaction = ANY(%s);
    """

    return run_query_jiwaplus(query, params=(list(transaction_ids),))


In [ ]:
def get_markassajiwa_user_ids():
    """Daftar user_id yang punya voucher membership 'Markas Sejuta Jiwa' aktif SAAT INI.

    Query persis dari spec yang diberikan, hanya :date diganti CURRENT_DATE sesuai arahan --
    artinya flag ini snapshot keanggotaan HARI INI (saat ETL dijalankan), bukan status
    keanggotaan di tanggal order historis dibuat.

    Asumsi (belum diverifikasi silang): user_invoice_voucher.user_id == customer_id di
    pending_orders / jiwaplus.transactions.customer_id -- keduanya sama-sama id akun JiwaPlus.
    """
    query = """
        SELECT DISTINCT(uiv.user_id)
        FROM user_invoice_voucher uiv
        WHERE uiv.voucher_id = %s
          AND uiv.active_at > CURRENT_DATE;
    """

    df = run_query_jiwaplus(query, params=(MARKASSAJIWA_VOUCHER_ID,))
    return set(df['user_id'].tolist())


In [ ]:
def get_blasted_rider_codes(pending_order_ids):
    """Daftar rider_code yang di-blast/di-ping untuk tiap order, diurutkan dari yang
    PALING DEKAT ke customer (distance_in_meter ASC, fallback waktu ping kalau distance NULL).

    Beda dengan versi standalone (etl_ksj_on_demand_blasted_rider_codes.ipynb) yang harus
    mapping lewat transaction_id (karena sumbernya Excel) -- di sini kita JOIN langsung
    pakai pending_order_id, karena sudah tersedia dari df_orders.
    """
    query = """
        SELECT
            pending_order_id,
            rider_code,
            distance_in_meter,
            created_at
        FROM dispatch_blast_riders
        WHERE pending_order_id = ANY(%s)
        ORDER BY pending_order_id, distance_in_meter ASC NULLS LAST, created_at ASC;
    """

    df = run_query(DB_DISPATCHER, query, params=(list(pending_order_ids),))
    if len(df) == 0:
        return pd.DataFrame(columns=['pending_order_id', 'blasted_rider_codes'])

    return (df.groupby('pending_order_id')['rider_code']
              .apply(lambda codes: ', '.join(codes))
              .reset_index()
              .rename(columns={'rider_code': 'blasted_rider_codes'}))


## Hitung jarak

In [ ]:
def haversine_m(lat1, lng1, lat2, lng2):
    """Jarak garis lurus (great-circle) dalam meter. R bumi = 6.371.000 m.

    Nilai lat/lng dari Postgres bertipe Decimal -> dipaksa ke float dulu.
    """
    vals = [lat1, lng1, lat2, lng2]
    if any(v is None or pd.isna(v) for v in vals):
        return np.nan

    lat1, lng1, lat2, lng2 = [float(v) for v in vals]
    R = 6371000.0

    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi       = np.radians(lat2 - lat1)
    dlambda    = np.radians(lng2 - lng1)

    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    return round(R * 2.0 * np.arcsin(np.sqrt(a)), 2)


In [ ]:
_route_cache = {}

def _osrm_route(lat1, lng1, lat2, lng2):
    """Panggil OSRM. Format URL-nya {lng},{lat} -- bukan {lat},{lng}."""
    url = "{0}/{1},{2};{3},{4}?overview=false".format(OSRM_BASE, lng1, lat1, lng2, lat2)
    for attempt in range(ROUTE_MAX_RETRY + 1):
        try:
            resp = requests.get(url, timeout=ROUTE_TIMEOUT)
            data = resp.json()
            if data.get('code') == 'Ok' and data.get('routes'):
                route = data['routes'][0]
                return (round(float(route['distance']), 2),
                        round(float(route['duration']), 1),
                        'OSRM')
            return (None, None, 'NO_ROUTE')
        except Exception as e:
            if attempt == ROUTE_MAX_RETRY:
                print('   OSRM gagal (%s,%s)->(%s,%s): %s' % (lat1, lng1, lat2, lng2, type(e).__name__))
                return (None, None, 'FAILED')
            time.sleep(1.0)
    return (None, None, 'FAILED')


def _google_route(lat1, lng1, lat2, lng2):
    """Panggil Google Directions API. Butuh GOOGLE_MAPS_API_KEY."""
    if not GOOGLE_MAPS_API_KEY:
        return (None, None, 'NO_API_KEY')

    url = 'https://maps.googleapis.com/maps/api/directions/json'
    params = {'origin': '%s,%s' % (lat1, lng1),
              'destination': '%s,%s' % (lat2, lng2),
              'mode': 'driving',
              'key': GOOGLE_MAPS_API_KEY}
    for attempt in range(ROUTE_MAX_RETRY + 1):
        try:
            data = requests.get(url, params=params, timeout=ROUTE_TIMEOUT).json()
            if data.get('status') == 'OK' and data.get('routes'):
                leg = data['routes'][0]['legs'][0]
                return (round(float(leg['distance']['value']), 2),
                        round(float(leg['duration']['value']), 1),
                        'GOOGLE')
            return (None, None, 'NO_ROUTE')
        except Exception as e:
            if attempt == ROUTE_MAX_RETRY:
                print('   Google gagal: %s' % type(e).__name__)
                return (None, None, 'FAILED')
            time.sleep(1.0)
    return (None, None, 'FAILED')


def get_route_distance(lat1, lng1, lat2, lng2):
    """Jarak rute jalan sebenarnya + estimasi durasi.

    Hasilnya di-cache per pasangan koordinat supaya order dengan titik yang sama
    (misal kantor pusat Kedoya yang sering jadi alamat antar) tidak dipanggil berulang.

    Return: (distance_m, duration_seconds, source)
    """
    vals = [lat1, lng1, lat2, lng2]
    if any(v is None or pd.isna(v) for v in vals):
        return (None, None, 'NO_COORDINATE')

    key = tuple(round(float(v), 6) for v in vals)
    if key in _route_cache:
        return _route_cache[key]

    if ROUTE_PROVIDER == 'OSRM':
        result = _osrm_route(*key)
    elif ROUTE_PROVIDER == 'GOOGLE':
        result = _google_route(*key)
    else:
        result = (None, None, 'DISABLED')

    _route_cache[key] = result
    time.sleep(ROUTE_SLEEP_SECONDS)
    return result


# GET DATE

In [ ]:
# Edit start_date / end_date di sini kalau mau backfill atau re-run tanggal tertentu.
# Default di bawah = kemarin (H-1), sesuai jadwal harian ETL ini (dijalankan hari ini,
# memproses data kemarin -- data hari ini belum selesai jadi tidak pernah ikut ke-dump).
#
# NOTE: pakai date(...) langsung (bukan datetime.date(...)) -- cell IMPORT LIBRARY di atas
# sudah "from datetime import datetime, timedelta, date", jadi nama `datetime` sudah
# ketiban jadi class datetime.datetime, bukan lagi module datetime.

yesterday = date.today() - timedelta(days=1)

start_date = date(2026, 8, 3)
end_date   = date(2026, 9, 10)

start_date_str = start_date.strftime('%Y-%m-%d')
end_date_str   = end_date.strftime('%Y-%m-%d')

print("start_date:", start_date_str)
print("end_date  :", end_date_str)


# EXTRACT

In [ ]:
df_orders        = get_pending_orders(start_date_str, end_date_str)
df_blast_riders  = get_blast_rider_metrics(start_date_str, end_date_str)
df_blast_rounds  = get_blast_round_metrics(start_date_str, end_date_str)
df_accept_ctx    = get_accept_context(start_date_str, end_date_str)
df_nearest       = get_nearest_blasted_rider(start_date_str, end_date_str)
df_delivery      = get_delivery_detail(start_date_str, end_date_str)

print('pending_orders     :', len(df_orders))
display(df_orders.head(3))

print('blast rider metrics:', len(df_blast_riders))
display(df_blast_riders.head(3))

print('blast round metrics:', len(df_blast_rounds))
display(df_blast_rounds.head(3))

print('accept context     :', len(df_accept_ctx))
display(df_accept_ctx.head(3))

print('nearest rider      :', len(df_nearest))
display(df_nearest.head(3))

print('delivery detail    :', len(df_delivery))
display(df_delivery.head(3))


## Extract tambahan (butuh daftar transaction_id / pending_order_id dari df_orders)

In [ ]:
df_jiwaplus_detail  = get_jiwaplus_transaction_detail(df_orders['transaction_id'].dropna().tolist())
df_blasted_codes    = get_blasted_rider_codes(df_orders['pending_order_id'].tolist())
df_blast_positions  = get_blast_rider_positions(df_orders['pending_order_id'].tolist())
df_status_detail    = get_first_dispatch_reason(df_orders['pending_order_id'].tolist())
markassajiwa_user_ids = get_markassajiwa_user_ids()

print('jiwaplus transaction detail :', len(df_jiwaplus_detail))
display(df_jiwaplus_detail.head(3))

print('blasted rider codes         :', len(df_blasted_codes))
display(df_blasted_codes.head(3))

print('posisi tiap rider di-blast   :', len(df_blast_positions), '(dipakai utk hitung rute per rider)')
display(df_blast_positions.head(3))

print('order_status_detail (reason paling awal) :', len(df_status_detail))
print(df_status_detail['order_status_detail'].value_counts().to_string())

print('total user_id have_markassajiwa (voucher %s aktif hari ini) :' % MARKASSAJIWA_VOUCHER_ID,
      len(markassajiwa_user_ids))


# TRANSFORM

## Gabung semua sumber

In [ ]:
# Ketiga database terpisah -> penggabungan dilakukan di sini, bukan di SQL.
df = df_orders.copy()

for part in [df_blast_riders, df_blast_rounds, df_accept_ctx, df_nearest]:
    df = df.merge(part, on='pending_order_id', how='left')

df_dlv = df_delivery.drop_duplicates(subset=['pending_order_id'], keep='last')
df = df.merge(df_dlv, on='pending_order_id', how='left')

count_cols = ['blast_count', 'total_riders_blasted', 'riders_accepted', 'riders_rejected',
              'riders_timeout', 'riders_no_response', 'riders_wa_sent', 'riders_push_sent',
              'total_blast_rounds', 'last_blast_round', 'total_riders_notified',
              'accepted_blast_round', 'riders_blasted_until_accept']
for col in count_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

print('rows setelah merge:', len(df), '(harus sama dengan jumlah pending_orders)')
assert len(df) == len(df_orders), 'jumlah baris berubah setelah merge -> ada duplikat di tabel pendukung'
display(df.head(3))


## Tentukan titik asal (origin) tiap order

In [ ]:
# Order ACCEPTED -> titik asal = posisi rider yang menerima order.
# Order gagal     -> tidak ada rider yang accept, jadi tidak ada titik asal yang pasti.
#                    Dipakai posisi rider TERDEKAT yang di-blast sebagai proksi, supaya
#                    kolom jarak tetap terisi untuk case failed (sesuai kebutuhan analisa).
#                    Interpretasinya: "rider terdekat yang tersedia pun jaraknya berapa".
#
# CATATAN DATA: kolom blast_rider_lat/long dan distance_in_meter di dispatch_blast_riders
# baru mulai diisi sistem tanggal 11 Agustus 2026. Order sebelum itu (dan sebagian order
# 11 Agt) tidak punya koordinat rider sama sekali -> origin-nya memang tidak bisa dihitung,
# ditandai origin_type = 'NO_RIDER_COORDINATE' (bukan error, murni keterbatasan data lama).

is_accepted  = df['dispatcher_status'].eq('ACCEPTED') & df['accepted_rider_lat'].notna()
has_nearest  = df['nearest_blasted_rider_lat'].notna()
no_blast     = df['total_riders_blasted'].eq(0)

df['origin_lat'] = np.where(is_accepted, df['accepted_rider_lat'],
                            np.where(has_nearest, df['nearest_blasted_rider_lat'], np.nan))
df['origin_lng'] = np.where(is_accepted, df['accepted_rider_lng'],
                            np.where(has_nearest, df['nearest_blasted_rider_lng'], np.nan))
df['origin_rider_code'] = np.where(is_accepted, df['accepted_rider_code'],
                                   np.where(has_nearest, df['nearest_blasted_rider_code'], None))
df['origin_type'] = np.select(
    [is_accepted, has_nearest, no_blast],
    ['ACCEPTED_RIDER', 'NEAREST_BLASTED_RIDER', 'NO_RIDER_BLASTED'],
    default='NO_RIDER_COORDINATE')

df['origin_lat'] = pd.to_numeric(df['origin_lat'], errors='coerce')
df['origin_lng'] = pd.to_numeric(df['origin_lng'], errors='coerce')

print(df['origin_type'].value_counts().to_string())
print()
print(pd.crosstab(df['dispatcher_status'], df['origin_type']))


## Straight line distance (haversine)

In [ ]:
# Kolom ini TETAP garis lurus (haversine) dengan sengaja -- namanya "straight_line" dan
# dipakai sebagai pembanding untuk route_detour_ratio, jadi tidak diubah jadi rute OSRM
# seperti kolom jarak lainnya (kalau ikut diganti, detour ratio jadi tidak ada artinya lagi).
df['straight_line_distance_m'] = df.apply(
    lambda r: haversine_m(r['origin_lat'], r['origin_lng'],
                          r['delivery_lat'], r['delivery_lng']), axis=1)

print(df.groupby('origin_type')['straight_line_distance_m'].describe()[['count', 'mean', 'min', 'max']])


## Actual route distance (routing API)

In [ ]:
# Dipanggil hanya untuk pasangan koordinat yang unik -> hemat request.
# Kalau API gagal, kolomnya NULL dan route_source='FAILED', proses tetap lanjut.
t0 = time.time()

routes = df.apply(lambda r: get_route_distance(r['origin_lat'], r['origin_lng'],
                                               r['delivery_lat'], r['delivery_lng']), axis=1)

df['actual_route_distance_m'] = [x[0] for x in routes]
df['route_duration_seconds']  = [x[1] for x in routes]
df['route_source']            = [x[2] for x in routes]

df['route_detour_ratio'] = (pd.to_numeric(df['actual_route_distance_m'], errors='coerce') /
                            pd.to_numeric(df['straight_line_distance_m'], errors='coerce')
                              .where(lambda s: s > 0)).round(3)

print('selesai dalam %.1f detik | %s pasangan koordinat unik dipanggil' % (time.time() - t0, len(_route_cache)))
print(df['route_source'].value_counts().to_string())
print('\ndetour ratio -> median: %s | min: %s | max: %s' % (
    df['route_detour_ratio'].median(), df['route_detour_ratio'].min(), df['route_detour_ratio'].max()))


## Jarak rute (OSRM) untuk SETIAP rider yang di-blast -- termasuk order yang GAGAL

In [ ]:
# Sebelumnya nearest/avg/max_blasted_rider_distance_m dihitung dari distance_in_meter
# (garis lurus yang dihitung SISTEM dispatcher, blast position -> delivery). Sekarang
# diganti jarak RUTE JALAN (OSRM) per rider individual -- dihitung untuk SEMUA order,
# baik yang ACCEPTED maupun EXPIRED/CANCELLED (rider tidak accept), bukan cuma yang sukses.
#
# Pakai cache & fungsi routing yang sama dengan cell sebelumnya (_route_cache, get_route_distance)
# supaya pasangan koordinat yang sudah pernah dipanggil (mis. titik delivery yang sama
# dipakai berkali-kali) tidak dipanggil ulang ke API.
df_blast_positions = df_blast_positions.merge(
    df_orders[['pending_order_id', 'delivery_lat', 'delivery_lng']], on='pending_order_id', how='left')

t0 = time.time()
blast_routes = df_blast_positions.apply(
    lambda r: get_route_distance(r['blast_rider_lat'], r['blast_rider_long'],
                                 r['delivery_lat'], r['delivery_lng']), axis=1)
df_blast_positions['route_distance_m'] = [x[0] for x in blast_routes]

print('selesai dalam %.1f detik | %s pasangan koordinat unik di cache (kumulatif)' % (
    time.time() - t0, len(_route_cache)))

blast_route_agg = df_blast_positions.groupby('pending_order_id')['route_distance_m'].agg(
    nearest_blasted_rider_distance_m='min',
    avg_blasted_rider_distance_m='mean',
    median_blasted_rider_distance_m='median',
    max_blasted_rider_distance_m='max',
).round(2).reset_index()

print('order dgn minimal 1 rider ter-routing :', len(blast_route_agg))
display(blast_route_agg.head(5))


## Gabung sumber tambahan (jiwaplus, blasted_rider_codes, jarak rute per rider, markassajiwa)

In [ ]:
df = df.merge(df_jiwaplus_detail, left_on='transaction_id', right_on='id_transaction', how='left')
df = df.merge(df_blasted_codes, on='pending_order_id', how='left')
df = df.merge(blast_route_agg, on='pending_order_id', how='left')
df = df.merge(df_status_detail, on='pending_order_id', how='left')

# latest_transaction_status dibiarkan RAW apa adanya (ORDER_COMPLETED/ORDER_CANCELLED,
# uppercase) -- sama seperti order_status, tidak ada transformasi/lowercase lagi.

# delivered_distance_m: dulu haversine (origin -> rider_completion_lat/lng), sekarang rute OSRM --
# jarak jalan sebenarnya dari titik asal sampai rider menekan tombol selesai antar.
df['delivered_distance_m'] = df.apply(
    lambda r: get_route_distance(r['origin_lat'], r['origin_lng'],
                                 r['rider_completion_lat'], r['rider_completion_lng'])[0], axis=1)

df['have_markassajiwa'] = df['customer_id'].isin(markassajiwa_user_ids)

print('baris tanpa data jiwaplus (transaction_id tidak ketemu) :', df['id_transaction'].isna().sum())
print('baris tanpa rider yang di-blast (blasted_rider_codes kosong) :', df['blasted_rider_codes'].isna().sum())
print('baris tanpa jarak rute per-rider (tidak ada koordinat sama sekali) :', df['nearest_blasted_rider_distance_m'].isna().sum())


## Kolom turunan & finalisasi

In [ ]:
# order_status = PERSIS pending_orders.status apa adanya (ACCEPTED/EXPIRED/CANCELLED),
# tidak di-mapping lagi ke SUCCESS/FAILED -- sudah dikonfirmasi cuma 3 value ini di DB.
df['order_status'] = df['dispatcher_status']

# order_status_detail sudah datang dari get_first_dispatch_reason() (merge di atas) --
# TIDAK ada hardcode/derivasi lagi, isinya persis dispatch_records.reason yang paling awal.
# Order CANCELLED yang rider-nya sempat accept (mis. id 5/11/12) otomatis benar kelabel
# 'rider accepted', bukan 'Rider Not Accepted' seperti versi hardcode sebelumnya.

df['created_at_wib']   = pd.to_datetime(df['created_at_wib'])
df['accepted_at_wib']  = pd.to_datetime(df['accepted_at_wib'])
df['delivered_at_wib'] = pd.to_datetime(df['delivered_at_wib'])
df['out_for_delivery_at_wib'] = pd.to_datetime(df['out_for_delivery_at_wib'])
df['paid_at_wib']            = pd.to_datetime(df['paid_at_wib'])
df['trx_completed_at_wib']   = pd.to_datetime(df['trx_completed_at_wib'])

df['order_date'] = df['created_at_wib'].dt.date
df['order_hour'] = df['created_at_wib'].dt.hour

# Flow lengkap 1 order: paid_at -> ... -> accepted_at -> out_for_delivery_at ("OTW", rider
# berangkat menuju customer) -> delivered_at/completed_at (rider sampai & upload bukti antar).
# Segmen-segmen di bawah menjawab pertanyaan berbeda-beda. Belum ada penanganan khusus untuk
# outlier (ada 1 order dgn minutes_paid_to_completed ~1549 menit / 25.8 jam) -- dibiarkan
# apa adanya sesuai arahan, difilter manual saat analisis kalau perlu.
df['minutes_paid_to_completed']   = ((df['trx_completed_at_wib'] - df['paid_at_wib'])
                                     .dt.total_seconds() / 60).round(2)
df['minutes_accept_to_otw']       = ((df['out_for_delivery_at_wib'] - df['accepted_at_wib'])
                                     .dt.total_seconds() / 60).round(2)
df['minutes_otw_to_delivered']    = ((df['delivered_at_wib'] - df['out_for_delivery_at_wib'])
                                     .dt.total_seconds() / 60).round(2)
df['minutes_accept_to_delivered'] = ((df['delivered_at_wib'] - df['accepted_at_wib'])
                                     .dt.total_seconds() / 60).round(2)

# Kolom final -- persis spec yang disepakati. Kolom lama yang tidak dipakai lagi
# (transaction_no, delivery_status, failed_reason, rincian respons rider per jenis,
# timestamp mentah created_at_wib dkk) SENGAJA dibuang dari output final.
RENAME_MAP = {
    'total_item_qty':             'order_qty',
    'total_riders_blasted':       'total_rider_blasted',
    'riders_blasted_until_accept':'total_rider_blasted_until_accept',
}
df = df.rename(columns=RENAME_MAP)

FINAL_COLUMNS = [
    'transaction_id', 'transaction_datetime_gmt', 'order_date', 'order_hour', 'pending_order_id',
    'latest_transaction_status', 'order_status', 'order_status_detail', 'hub_code', 'customer_id',
    'customer_name', 'customer_phone', 'have_markassajiwa',
    'order_qty', 'total_price', 'payment_type',
    'minutes_paid_to_completed', 'seconds_to_accept', 'minutes_accept_to_otw',
    'minutes_otw_to_delivered', 'minutes_accept_to_delivered',
    'origin_type', 'origin_rider_code', 'origin_lat', 'origin_lng', 'delivery_lat', 'delivery_lng',
    'straight_line_distance_m', 'actual_route_distance_m', 'route_duration_seconds',
    'delivered_distance_m', 'nearest_blasted_rider_distance_m', 'avg_blasted_rider_distance_m',
    'median_blasted_rider_distance_m', 'max_blasted_rider_distance_m', 'route_source',
    'total_rider_blasted', 'total_rider_blasted_until_accept', 'total_blast_rounds',
    'accepted_blast_round', 'blasted_rider_codes',
]

df_summary = df[FINAL_COLUMNS].copy()

num_cols = ['total_price', 'straight_line_distance_m', 'actual_route_distance_m',
            'route_duration_seconds', 'delivered_distance_m',
            'nearest_blasted_rider_distance_m', 'avg_blasted_rider_distance_m',
            'median_blasted_rider_distance_m', 'max_blasted_rider_distance_m', 'seconds_to_accept',
            'minutes_paid_to_completed', 'minutes_accept_to_otw', 'minutes_otw_to_delivered',
            'minutes_accept_to_delivered', 'origin_lat', 'origin_lng', 'delivery_lat', 'delivery_lng']
for col in num_cols:
    df_summary[col] = pd.to_numeric(df_summary[col], errors='coerce')

print('final shape:', df_summary.shape)
display(df_summary.head(5))


## Ringkasan: rata-rata jarak per order, dipisah success vs failed

In [ ]:
# Inilah pertanyaan utamanya. Tabel di atas grain per-order, jadi tinggal di-AVG per outcome.
# Pakai df (bukan df_summary) supaya masih bisa lihat route_detour_ratio -- kolom itu
# berguna untuk QA tapi tidak diminta masuk ke tabel final.
ringkasan = df.groupby('order_status').agg(
    total_order              = ('pending_order_id',                       'count'),
    order_dgn_koordinat      = ('straight_line_distance_m',               'count'),
    avg_straight_line_m      = ('straight_line_distance_m',               'mean'),
    median_straight_line_m   = ('straight_line_distance_m',               'median'),
    avg_actual_route_m       = ('actual_route_distance_m',                'mean'),
    median_actual_route_m    = ('actual_route_distance_m',                'median'),
    # detour ratio pakai MEDIAN, bukan mean: ada order yang rider-nya sudah persis
    # di titik antar (garis lurus 5,9 m) sehingga rasionya meledak jadi 40x
    # dan bikin rata-rata tidak representatif.
    median_detour_ratio      = ('route_detour_ratio',                     'median'),
    avg_qty_per_order        = ('order_qty',                              'mean'),
    avg_riders_blasted       = ('total_rider_blasted',                    'mean'),
    avg_blast_rounds         = ('total_blast_rounds',                     'mean'),
    avg_seconds_to_accept    = ('seconds_to_accept',                      'mean'),
).round(2)

display(ringkasan)

print('\nDi-dispatch ke berapa rider sebelum ada yang accept (order ACCEPTED saja):')
display(df[df.order_status == 'ACCEPTED']
        .groupby('accepted_blast_round')
        .agg(total_order=('pending_order_id', 'count'),
             avg_riders_blasted_until_accept=('total_rider_blasted_until_accept', 'mean'),
             avg_seconds_to_accept=('seconds_to_accept', 'mean')).round(2))


## Sanity check

In [ ]:
# Angka acuan di bawah ini sudah diverifikasi langsung ke database saat ETL ini dibuat.
# Kalau ada yang BEDA, jangan di-dump dulu.

def _chk(label, actual, expected):
    status = 'OK  ' if actual == expected else 'BEDA'
    print('[%s] %-52s actual=%s expected=%s' % (status, label, actual, expected))

_chk('grain: pending_order_id unik',
     int(df_summary['pending_order_id'].duplicated().sum()), 0)
# order_ttl_minutes tidak ikut di kolom final, tapi tetap ada di df (internal) -- dicek di situ.
# CATATAN: sejak 29 Agustus 2026 TTL berubah dari 6 menit -> 9 menit (perubahan bisnis
# yang sah, bukan bug) -- jadi ini INFO distribusi, bukan assert nilai tunggal lagi.
print('[INFO] distribusi order_ttl_minutes (perubahan aturan bisnis akan kelihatan di sini):')
print(df.groupby(df['order_ttl_minutes'].round(2))['pending_order_id']
        .agg(['count', 'min', 'max']).rename(columns={'min': 'contoh_id_min', 'max': 'contoh_id_max'}))
_chk('order_status hanya ACCEPTED/EXPIRED/CANCELLED (value RAW pending_orders.status)',
     sorted(df_summary['order_status'].dropna().unique().tolist()),
     sorted(['ACCEPTED', 'EXPIRED', 'CANCELLED']))
_chk('latest_transaction_status hanya ORDER_COMPLETED/ORDER_CANCELLED (value RAW jiwaplus)',
     sorted(df_summary['latest_transaction_status'].dropna().unique().tolist()),
     sorted(['ORDER_COMPLETED', 'ORDER_CANCELLED']))
# order_status_detail = reason mentah dispatch_records apa adanya, TIDAK di-hardcode lagi.
_chk('order_status_detail hanya 5 value mentah yang dikonfirmasi ke DB',
     sorted(df_summary['order_status_detail'].dropna().unique().tolist()),
     sorted(['rider accepted', 'expired - no rider accepted', 'no nearby rider',
            'no rider with sufficient stock',
            'no eligible rider within road distance / with sufficient stock']))
_chk('have_markassajiwa harus boolean, tidak ada yang null',
     int(df_summary['have_markassajiwa'].isna().sum()), 0)

if 52 in set(df_summary['pending_order_id']):
    o52 = df_summary[df_summary.pending_order_id == 52].iloc[0]
    o52_df = df[df.pending_order_id == 52].iloc[0]   # kolom internal yg tidak di-export
    _chk('order 52 - jiwaplus_ref_id (cek internal)', o52_df['jiwaplus_ref_id'],       'SJ-2608167536GN')
    _chk('order 52 - total_rider_blasted',    int(o52['total_rider_blasted']),       7)
    _chk('order 52 - riders_accepted (cek internal)', int(o52_df['riders_accepted']), 1)
    _chk('order 52 - seconds_to_accept',      round(float(o52['seconds_to_accept']), 1), 56.5)
    _chk('order 52 - origin_type',            o52['origin_type'],                    'ACCEPTED_RIDER')
    # 463 m = haversine dari posisi rider saat ACCEPT (pending_orders.rider_lat/long).
    # Jangan bandingkan dengan dispatch_blast_riders.distance_in_meter yang 459,49 m --
    # angka itu dihitung dari posisi rider saat DI-BLAST, titik yang sedikit berbeda.
    _chk('order 52 - straight_line (bulat)',  int(round(float(o52['straight_line_distance_m']))), 463)
    # accepted 05:09:53.971 -> otw 05:13:52.010 -> delivered 05:20:23.224 (WIB)
    _chk('order 52 - minutes_accept_to_otw',       round(float(o52['minutes_accept_to_otw']), 2),       3.97)
    _chk('order 52 - minutes_otw_to_delivered',    round(float(o52['minutes_otw_to_delivered']), 2),    6.52)
    _chk('order 52 - minutes_accept_to_delivered', round(float(o52['minutes_accept_to_delivered']), 2), 10.49)
    _chk('order 52 - minutes_paid_to_completed',   round(float(o52['minutes_paid_to_completed']), 2),   11.44)
    # jiwaplus.transactions.id_transaction=7907522 -- BUKAN d_transaction.transactions.id=4359112
    _chk('order 52 - customer_name',               o52['customer_name'],             'Linda Wulanika')
    _chk('order 52 - latest_transaction_status',    o52['latest_transaction_status'], 'ORDER_COMPLETED')
    _chk('order 52 - order_status',                 o52['order_status'],              'ACCEPTED')
    _chk('order 52 - order_status_detail',          o52['order_status_detail'],       'rider accepted')
    _chk('order 52 - blasted_rider_codes (urut terdekat->terjauh)', o52['blasted_rider_codes'],
         'JW00202, GPI000379, TGC31179, SKN4556, JW00243, GPI03710, GPI02765')
    # jarak RUTE (OSRM) dari posisi blast tiap rider -> delivery, dihitung manual via OSRM
    # langsung ke database saat kolom ini ditambahkan (7 rider, lihat komentar di FUNCTION).
    _chk('order 52 - nearest_blasted_rider_distance_m (rute)', round(float(o52['nearest_blasted_rider_distance_m']), 2), 813.40)
    _chk('order 52 - avg_blasted_rider_distance_m (rute)',     round(float(o52['avg_blasted_rider_distance_m']), 2),     5761.44)
    _chk('order 52 - median_blasted_rider_distance_m (rute)',  round(float(o52['median_blasted_rider_distance_m']), 2),  6075.50)
    _chk('order 52 - max_blasted_rider_distance_m (rute)',     round(float(o52['max_blasted_rider_distance_m']), 2),     9524.70)
    _chk('order 52 - delivered_distance_m (rute, bukan haversine lagi)', round(float(o52['delivered_distance_m']), 2), 619.20)

if 40 in set(df_summary['pending_order_id']):
    o40 = df_summary[df_summary.pending_order_id == 40].iloc[0]
    o40_df = df[df.pending_order_id == 40].iloc[0]
    _chk('order 40 - total_blast_rounds',     int(o40['total_blast_rounds']),        3)
    _chk('order 40 - total_rider_blasted',    int(o40['total_rider_blasted']),       22)
    # 3 REJECTED + 19 TIMEOUT = 22 ping. Kalau angkanya keluar 9 dan 57, berarti
    # dispatch_blasts dan dispatch_blast_riders ke-JOIN bareng -> cartesian product.
    _chk('order 40 - riders_rejected (cek internal)', int(o40_df['riders_rejected']), 3)
    _chk('order 40 - riders_timeout (cek internal)',  int(o40_df['riders_timeout']),  19)
    _chk('order 40 - order_status',           o40['order_status'],                   'EXPIRED')
    _chk('order 40 - order_status_detail',    o40['order_status_detail'],            'expired - no rider accepted')
    _chk('order 40 - origin_type',            o40['origin_type'],                    'NEAREST_BLASTED_RIDER')
    # order 40 GAGAL (tidak ada rider accept) -- jarak rute per-rider ini yang membuktikan
    # poin ke-2: distance sekarang tetap terisi walau ordernya rider not accept.
    _chk('order 40 - nearest_blasted_rider_distance_m (rute)', round(float(o40['nearest_blasted_rider_distance_m']), 2), 3936.90)
    _chk('order 40 - avg_blasted_rider_distance_m (rute)',     round(float(o40['avg_blasted_rider_distance_m']), 2),     8335.72)
    _chk('order 40 - median_blasted_rider_distance_m (rute)',  round(float(o40['median_blasted_rider_distance_m']), 2),  8713.90)
    _chk('order 40 - max_blasted_rider_distance_m (rute)',     round(float(o40['max_blasted_rider_distance_m']), 2),     10871.40)
    # order 40 origin_type = NEAREST_BLASTED_RIDER -> actual_route_distance_m (origin->delivery)
    # HARUS sama dengan nearest_blasted_rider_distance_m (sama-sama rider terdekat, titik sama).
    _chk('order 40 - actual_route_distance_m == nearest_blasted_rider_distance_m',
         round(float(o40['actual_route_distance_m']), 2), round(float(o40['nearest_blasted_rider_distance_m']), 2))

# order 5/11/12: CANCELLED, tapi rider SEMPAT accept (accepted_rider_code terisi) sebelum
# order dibatalkan. Ini pembuktian utama kenapa order_status_detail tidak boleh hardcode
# dari order_status lagi -- versi lama salah label ketiganya jadi 'Rider Not Accepted'.
for oid in [5, 11, 12]:
    if oid in set(df_summary['pending_order_id']):
        o = df_summary[df_summary.pending_order_id == oid].iloc[0]
        _chk(f'order {oid} - order_status (CANCELLED, bukan gagal cari rider)', o['order_status'], 'CANCELLED')
        _chk(f'order {oid} - order_status_detail (rider SEMPAT accept)', o['order_status_detail'], 'rider accepted')

# rekonsiliasi jumlah baris dengan tabel sumber
recon = run_query(DB_DISPATCHER, """
    SELECT COUNT(*) AS n FROM pending_orders
    WHERE (created_at AT TIME ZONE 'Asia/Jakarta')::date BETWEEN '{0}' AND '{1}'
""".format(start_date_str, end_date_str))
_chk('jumlah baris = jumlah pending_orders di window',
     len(df_summary), int(recon['n'].iloc[0]))

# routing tidak boleh gagal total
n_ok = int((df_summary['route_source'] == ROUTE_PROVIDER).sum())
n_need = int(df_summary['straight_line_distance_m'].notna().sum())
print('[INFO] routing berhasil %s dari %s baris yang punya koordinat lengkap' % (n_ok, n_need))
if n_need and n_ok < n_need * 0.8:
    print('[WARN] lebih dari 20% routing gagal -- cek koneksi / rate limit sebelum dump')


# LOAD

## Engine

In [ ]:
engine = create_engine("mysql+pymysql://{user}:{pw}@{hostsname}/{db}".format(
    hostsname=config_dl_dwh.hostname,
    user=config_dl_dwh.username,
    pw=config_dl_dwh.password,
    db=config_dl_dwh.db_target
))


## Hapus dulu range tanggalnya (biar aman kalau notebook di-rerun)

In [ ]:
from sqlalchemy import text, inspect

delete_sql = text("DELETE FROM {0} WHERE order_date BETWEEN :start_date AND :end_date;".format(TABLE_ORDER_SUMMARY))

existing_tables = inspect(engine).get_table_names()

with engine.begin() as conn:
    if TABLE_ORDER_SUMMARY in existing_tables:
        res = conn.execute(delete_sql, {"start_date": start_date_str, "end_date": end_date_str})
        print('%s : %s baris dihapus' % (TABLE_ORDER_SUMMARY, res.rowcount))
    else:
        print('%s : belum ada, akan dibuat otomatis oleh to_sql' % TABLE_ORDER_SUMMARY)


## Dump to DWH

In [ ]:
df_summary.to_sql(
    name=TABLE_ORDER_SUMMARY,
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=500
)

print('dumped %s rows -> %s' % (len(df_summary), TABLE_ORDER_SUMMARY))


## Verifikasi hasil dump

In [ ]:
with engine.connect() as conn:
    chk = pd.read_sql(text("""
        SELECT order_status,
               COUNT(*)                                     AS total_order,
               ROUND(AVG(straight_line_distance_m), 2)      AS avg_straight_line_m,
               ROUND(AVG(actual_route_distance_m), 2)       AS avg_actual_route_m,
               ROUND(AVG(nearest_blasted_rider_distance_m), 2) AS avg_nearest_blasted_rider_m,
               ROUND(AVG(order_qty), 2)                     AS avg_qty,
               ROUND(AVG(total_rider_blasted), 2)           AS avg_riders_blasted
        FROM {0}
        WHERE order_date BETWEEN '{1}' AND '{2}'
        GROUP BY 1
    """.format(TABLE_ORDER_SUMMARY, start_date_str, end_date_str)), conn)

display(chk)
